In [1]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
from model.systemModel                  import SystemModel as systemModel
# from model.componentModel               import SystemModel as componentModel
from sklearn.preprocessing              import MinMaxScaler

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json


# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

2024-11-29 12:13:00.214201: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-29 12:13:00.651441: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-29 12:13:02.040719: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/ducan/anaconda3/envs/tensorflow2.10.0Crusher/lib/
2024-11-29 12:13:02.040923: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerro

##### HI COMPONENT AND SYSTEM

In [2]:
preData                 = PreProcessData(config = myConfig)                    # Preprocessing data

cr1, cr2, cv2           = preData.loadH5File(name = "WITHOUTNonCriticalData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file



# --------------------------------
# Add hour column into the file
# print(cv2.columns)
withHourCr1             = preData.addHours(cr1)
withHourCr2             = preData.addHours(cr2)
withHourCv2             = preData.addHours(cv2)


# --------------------------------
# Use for system HI
# --------------------------------
insertedCr1Sys, insertedCr2Sys,  insertedCv2Sys     = preData.insertForTheSameLength(data         = [withHourCr1.copy(),withHourCr2.copy(),withHourCv2.copy()], 
                                                                                    validColumn  = [preData.crusher1Value, preData.crusher2Value, preData.conveyorValue + preData.crusher2ConveyorValue])

withIDCr1, withIDCr2, withIDCv2, IdSys  = preData.attachIDPerEvent(df = [insertedCr1Sys,insertedCr2Sys,insertedCv2Sys], mainternanceAsCriticalSd = True)

# insertedCr1 = withHourCr1
# insertedCr2 = withHourCr2
# insertedCv2 = withHourCv2

# withIDCr1, withIDCr2, withIDCv2, IdSys = preData.attachIDPerEvent(df = [insertedCr1,insertedCr2,insertedCv2], mainternanceAsCriticalSd = True)


addedData               = preData.addition_var(withIDCr1) 


handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1
# -----------------------------
# Normalize
normalizedCr1           = preData.normalizeBuilinFunction(addedData, headline = handledValues)  
normalizedCr2           = preData.normalizeBuilinFunction(withIDCr2, headline = preData.crusher2Value)  
normalizedCv2           = preData.normalizeBuilinFunction(withIDCv2, headline = preData.conveyorValue + preData.crusher2Value)  

# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr1         = preData.addCycleAndRul(normalizedCr1)                        # Add the time to the data
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2)                        # Add the time to the data
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2)                        # Add the time to the data

# -----------------------------
# Add the cycle and RUL of system
systemData                  = pd.DataFrame()
systemData['hours']         = withTimeDataCr1["hours"]
systemData['trajectoryID']  = IdSys
withTimeDataSys             = preData.addCycleAndRul(systemData)

# -----------------------------
# Extract before splitting
dataCr1                 = withTimeDataCr1[handledValues]
cycleCr1                = withTimeDataCr1["cycle"]
rulCr1                  = withTimeDataCr1["rul"]
timeCr1                 = withTimeDataCr1["hours"]
trajectIDCr1            = withTimeDataCr1["trajectoryID"]

dataCr2                 = withTimeDataCr2[preData.crusher2Value]
cycleCr2                = withTimeDataCr2["cycle"]
rulCr2                  = withTimeDataCr2["rul"]
timeCr2                 = withTimeDataCr2["hours"]
trajectIDCr2            = withTimeDataCr2["trajectoryID"]

dataCv2                 = withTimeDataCv2[preData.conveyorValue + preData.crusher2ConveyorValue]
cycleCv2                = withTimeDataCv2["cycle"]
rulCv2                  = withTimeDataCv2["rul"]
timeCv2                 = withTimeDataCv2["hours"]
trajectIDCv2            = withTimeDataCv2["trajectoryID"]

systemRUL               = withTimeDataSys["rul"]


print(timeCr1)
print(timeCr2)
print(timeCv2)
print(insertedCr1Sys["hours"])



Process:26383||2024-11-29 12:13:04,308-INFO [preProcessingData.py:loadH5File():575]:Loading H5 file ...
Process:26383||2024-11-29 12:13:05,728-INFO [preProcessingData.py:loadH5File():581]:Finish loading data from files ...
Process:26383||2024-11-29 12:13:09,454-INFO [preProcessingData.py:attachIDPerEvent():422]:Attaching ID into trajectories
Process:26383||2024-11-29 12:13:30,040-INFO [preProcessingData.py:normalizeBuilinFunction():1439]:Normalizing the data ....
Process:26383||2024-11-29 12:13:30,902-INFO [preProcessingData.py:normalizeBuilinFunction():1439]:Normalizing the data ....
Process:26383||2024-11-29 12:13:31,318-INFO [preProcessingData.py:normalizeBuilinFunction():1439]:Normalizing the data ....


0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dty

###### Crusher1

In [3]:
model       = systemModel(myConfig) 
model.build(input_shape=[(None, myConfig["seqLength"], myConfig["inCr1"]),(None, myConfig["seqLength"], myConfig["inCr2"]), (None, myConfig["seqLength"], myConfig["inCv2"])])
model.summary()
model.load_weights(model_dir = "checkpoint/bestModel43/")

2024-11-29 12:13:32.322542: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 12:13:32.322824: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 12:13:32.352220: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 12:13:32.352431: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-29 12:13:32.352619: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from S

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)


Process:26383||2024-11-29 12:13:34,162-INFO [systemModel.py:loadWeightAndFreezePretrainedHI():984]:Loading the model at the direction: checkpoint/bestModel44/


The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)


Process:26383||2024-11-29 12:13:34,216-INFO [systemModel.py:loadWeightAndFreezePretrainedHI():1023]:Loaded the model at the direction:checkpoint/bestModel44/ successfully!!!


Model: "system_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection (SensorSel  multiple                 0 (unused)
 ection)                                                         
                                                                 
 sensor_selection_1 (SensorS  multiple                 0 (unused)
 election)                                                       
                                                                 
 sensor_selection_2 (SensorS  multiple                 0 (unused)
 election)                                                       
                                                                 
 encoder (Functional)        [(None, 1),               122306    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 41)           

Process:26383||2024-11-29 12:13:35,377-INFO [systemModel.py:load_weights():947]:Loading the model at the direction: checkpoint/bestModel43/
Process:26383||2024-11-29 12:13:35,432-INFO [systemModel.py:load_weights():981]:Loaded the model at the direction:checkpoint/bestModel43/ successfully!!!


###### System

In [4]:
results            = {}
results["cr1"]     = []
results["cr2"]     = []
results["cv2"]     = []
results["sys"]     = []
results["rul"]     = []
listRUL         = []


probsAll        = []
binaryMaskAll   = []
trajectID = IdSys.astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr1,seqDataCr2,seqDataCv2, seqRUL     = preData.getTensorData(dataList   =    [dataCr1, dataCr2, dataCv2, systemRUL], 
                                                                    trajectId   = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, zCr2,  zCv2,                              \
    binaryMaskCr1, binaryMaskCr2, binaryMaskCv2,    \
    probsMaskCr1, probsMaskCr2, probsMaskCv2,       \
    hiSystem, predictedRul                          = model.getHIandRUL([seqDataCr1,seqDataCr2,seqDataCv2])
    results["cr1"].append(zCr1)
    results["cr2"].append(zCr2)
    results["cv2"].append(zCv2)
    results["sys"].append(hiSystem)
    results["rul"].append(predictedRul)
    listRUL.append(seqRUL[:,-1,:])


Process:26383||2024-11-29 12:13:35,447-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


0


2024-11-29 12:13:36.878502: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2024-11-29 12:13:37.706683: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-11-29 12:13:37.707152: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-11-29 12:13:37.707173: W tensorflow/stream_executor/gpu/asm_compiler.cc:80] Couldn't get ptxas version string: INTERNAL: Couldn't invoke ptxas --version
2024-11-29 12:13:37.707580: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-11-29 12:13:37.707632: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] INTERNAL: Failed to launch ptxas
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.
Process:26383||2024-11-29 12:13:38,771-INFO [preProcessingData.

1


Process:26383||2024-11-29 12:13:39,117-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


2


... (194 outputs left out)

Process:26383||2024-11-29 12:15:03,472-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


112


Process:26383||2024-11-29 12:15:05,220-INFO [preProcessingData.py:getTensorData():1213]:Spliting data into sequences now ...


113


In [5]:
def calculate_mae_for_numpy(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("The two lists must have the same length.")
    
    maes = []  # To store the MAE for each pair of arrays
    for arr1, arr2 in zip(list1, list2):
        if arr1.shape != arr2.shape:
            raise ValueError("Each pair of arrays must have the same shape.")
        mae = np.mean(np.abs(arr1 - arr2))  # Element-wise absolute difference, then mean
        maes.append(mae)
    
    overall_mae = np.mean(maes)  # Average MAE across all pairs
    return overall_mae

# Example usage

print(listRUL[0].shape, results["rul"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rul"])
print("Overall MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))

temp_df = pd.concat([rulCr1, rulCr2, rulCv2], axis=1)
# print(np.max(np.concatenate(rulCr1, rulCr2 , rulCv2)), np.min(np.concatenate(rulCr1 , rulCr2, rulCv2)))
print(temp_df.max().max(),temp_df.min().min())





(6123, 1) (6123, 1)
Overall MAE: 73.078224
454.7669999999998 0.0
670.55 0.0
